In [1]:
import sys, os
sys.path.append(os.path.abspath('..'))

import joblib
import pandas as pd
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from src.preprocessing import  get_preprocessor

c:\ai&machin\final_ml _project\telco-churn-mlops\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\ai&machin\final_ml _project\telco-churn-mlops\src\preprocessing.py:15: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas

In [2]:


df = pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
df.drop('customerID', axis=1, inplace=True)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')


X = df.drop('Churn', axis=1)
encoder_target = LabelEncoder()
y = encoder_target.fit_transform(df['Churn'])


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)




C:\Users\roky\AppData\Local\Temp\ipykernel_28008\3250225472.py:2: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop('customerID', axis=1, inplace=True)


In [3]:
TRACKING_URI = "sqlite:///mlruns.db"

mlflow.set_tracking_uri(TRACKING_URI)

print("Tracking URI:", mlflow.get_tracking_uri())



Tracking URI: sqlite:///mlruns.db


In [4]:
mlflow.set_tracking_uri("sqlite:///../mlflow.db")


In [5]:
db_path = os.path.abspath("../mlflow.db")
mlflow.set_tracking_uri(f"sqlite:///{db_path}")

In [6]:
experiment = mlflow.set_experiment("Telco_Churn_GridSearch")

In [7]:
models_and_grids = {
    "Logistic_Regression": (
        LogisticRegression(max_iter=1000, random_state=42),
        {
            'classifier__C': [0.1, 1.0, 10.0],
            'classifier__solver': ['liblinear', 'lbfgs']
        }
    ),
    "Random_Forest": (
        RandomForestClassifier(random_state=42),
        {
            'classifier__n_estimators':[100, 200] ,
            'classifier__max_depth': [5, 10, None],
            'classifier__min_samples_split': [2, 5]
        }
    ),
    "XGBoost": (
        XGBClassifier(eval_metric='logloss', random_state=42),
        {
            'classifier__n_estimators':[100, 200] ,
            'classifier__max_depth':[3, 5],
            'classifier__learning_rate': [0.05, 0.1]
        }
    )
}

In [9]:
results = []

In [10]:
for name, (model, param_grid) in models_and_grids.items():
    
    with mlflow.start_run(run_name=f"Tuned_{name}") as parent_run:
        pipeline = Pipeline(steps=[
            ('preprocessor', get_preprocessor(X_train)),
            ('classifier', model)
        ])
        
       
        grid_search = GridSearchCV(
            estimator=pipeline,
            param_grid=param_grid,
            cv=5,
            scoring={
                'roc_auc': 'roc_auc',
                'accuracy': 'accuracy',
                'precision': 'precision',
                'recall': 'recall',
                'f1': 'f1'
            },
            refit='roc_auc',
            n_jobs=-1
        )
        grid_search.fit(X_train, y_train)
        
       
        cv_results = pd.DataFrame(grid_search.cv_results_)
        for idx, row in cv_results.iterrows():
            with mlflow.start_run(run_name=f"{name}_trial_{idx+1}", nested=True):
                
                mlflow.log_params(row['params'])
                
                
                mlflow.log_metrics({
                    "cv_roc_auc": row['mean_test_roc_auc'],
                    "cv_accuracy": row['mean_test_accuracy'],
                    "cv_precision": row['mean_test_precision'],
                    "cv_recall": row['mean_test_recall'],
                    "cv_f1_score": row['mean_test_f1']
                })
        
        
        best_model = grid_search.best_estimator_
        y_pred = best_model.predict(X_test)
        y_prob = best_model.predict_proba(X_test)[:, 1]
        
        test_metrics = {
            "test_accuracy": accuracy_score(y_test, y_pred),
            "test_precision": precision_score(y_test, y_pred),
            "test_recall": recall_score(y_test, y_pred),
            "test_f1_score": f1_score(y_test, y_pred),
            "test_roc_auc": roc_auc_score(y_test, y_prob)
        }
        
        mlflow.log_params(grid_search.best_params_)
        mlflow.log_metrics(test_metrics)
        mlflow.sklearn.log_model(
            best_model,
            name=f"{name}_best_pipeline",
            serialization_format="cloudpickle"
        )
        
        results.append({
            "Model": name,
            **test_metrics,
            "Best_Params": str(grid_search.best_params_)
        })

2026/11/17 03:05:43 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/11/17 03:05:59 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/11/17 03:06:07 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mec

In [11]:
lr_params = {
    "C": 10,               
   
 
    "random_state": 42
}

In [12]:
rf_params = {
    "n_estimators": 200,     
    "min_samples_split": 5,  
    'max_depth':10,
    "random_state": 42
}

In [13]:
xgb_params = {
    "n_estimators": 100,      
    "max_depth": 3,        
    "learning_rate": 0.1,    
    "eval_metric": "logloss",
    "random_state": 42
}

In [14]:
final_models = {
    "Logistic_Regression": (LogisticRegression(**lr_params), lr_params),
    "Random_Forest": (RandomForestClassifier(**rf_params), rf_params),
    "XGBoost": (XGBClassifier(**xgb_params), xgb_params)
}

In [15]:
os.makedirs('../models', exist_ok=True)

In [16]:
for name, (model, params) in final_models.items():
    with mlflow.start_run(run_name=f"Final_Manual_{name}"):
        
        final_pipeline = Pipeline(steps=[
            ('preprocessor', get_preprocessor(X_train)),
            ('classifier', model)
        ])
        
        
        final_pipeline.fit(X_train, y_train)
        
       
        y_pred = final_pipeline.predict(X_test)
        y_prob = final_pipeline.predict_proba(X_test)[:, 1]
        
        metrics = {
            "accuracy": accuracy_score(y_test, y_pred),
            "precision": precision_score(y_test, y_pred),
            "recall": recall_score(y_test, y_pred),
            "f1_score": f1_score(y_test, y_pred),
            "roc_auc": roc_auc_score(y_test, y_prob)
        }
        
      
        mlflow.log_params(params)
        mlflow.log_metrics(metrics)
        mlflow.sklearn.log_model(
            final_pipeline,
            name=f"final_{name}_pipeline",
            serialization_format="cloudpickle"
        )
        
        
        file_path = f"../models/{name.lower()}_pipeline.joblib"
        joblib.dump(final_pipeline, file_path)
        print(f"Saved {name} -> {file_path} | Test ROC-AUC: {metrics['roc_auc']:.4f}")

2026/11/17 03:08:00 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Saved Logistic_Regression -> ../models/logistic_regression_pipeline.joblib | Test ROC-AUC: 0.8397


2026/11/17 03:08:07 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Saved Random_Forest -> ../models/random_forest_pipeline.joblib | Test ROC-AUC: 0.8414


2026/11/17 03:08:13 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Saved XGBoost -> ../models/xgboost_pipeline.joblib | Test ROC-AUC: 0.8437
